In [4]:
from pathlib import Path

REPO_URL = "https://github.com/Nikhil3654/generalizable-llm-planning.git"

WORKING_DIR = Path("/kaggle/working")
REPO_DIR = WORKING_DIR / "generalizable-llm-planning"
FD_DIR = WORKING_DIR / "downward"

DATASET_ROOT = None
NUM_BASELINE_PROBLEMS = 5

if "YOUR_GITHUB_USERNAME" in REPO_URL:
    raise ValueError(
        "Set REPO_URL to the public GitHub repository URL before continuing."
    )

In [5]:
import subprocess

if REPO_DIR.exists():
    print("Repository already exists. Pulling latest changes...")
    subprocess.run(
        ["git", "-C", str(REPO_DIR), "pull"],
        check=True,
    )
else:
    print("Cloning repository...")
    subprocess.run(
        ["git", "clone", REPO_URL, str(REPO_DIR)],
        check=True,
    )

print("Repository:", REPO_DIR)


Cloning repository...


Cloning into '/kaggle/working/generalizable-llm-planning'...


Repository: /kaggle/working/generalizable-llm-planning


In [6]:
import sys

FD_REPO_URL = "https://github.com/aibasel/downward.git"

if not FD_DIR.exists():
    subprocess.run(
        ["git", "clone", FD_REPO_URL, str(FD_DIR)],
        check=True,
    )

fast_downward_script = FD_DIR / "fast-downward.py"

if not fast_downward_script.exists():
    raise FileNotFoundError(f"Fast Downward entry point not found: {fast_downward_script}")

# Build once per fresh Kaggle session.
subprocess.run(
    [sys.executable, "build.py"],
    cwd=str(FD_DIR),
    check=True,
)

version = subprocess.run(
    [sys.executable, str(fast_downward_script), "--version"],
    capture_output=True,
    text=True,
    check=True,
)

print(version.stdout.strip() or version.stderr.strip())


Cloning into '/kaggle/working/downward'...


-- The C compiler identification is GNU 11.4.0
-- The CXX compiler identification is GNU 11.4.0
-- Detecting C compiler ABI info
-- Detecting C compiler ABI info - done
-- Check for working C compiler: /usr/bin/cc - skipped
-- Detecting C compile features
-- Detecting C compile features - done
-- Detecting CXX compiler ABI info
-- Detecting CXX compiler ABI info - done
-- Check for working CXX compiler: /usr/bin/c++ - skipped
-- Detecting CXX compile features
-- Detecting CXX compile features - done
-- Building for 64-bit.
-- Performing Test CMAKE_HAVE_LIBC_PTHREAD


Git revision: "9b81c7e42"


-- Performing Test CMAKE_HAVE_LIBC_PTHREAD - Success
-- Found Threads: TRUE
-- Could NOT find Cplex: Found unsuitable version "CPLEX_VERSION-NOTFOUND", but required is at least "12" (found CPLEX_INCLUDE_DIRS-NOTFOUND)
-- Configuring done (1.5s)
-- Generating done (0.1s)
-- Build files have been written to: /kaggle/working/downward/builds/release
Copying translator module into output directory
[  0%] Building CXX object search/CMakeFiles/downward.dir/planner.cc.o
[  1%] Building CXX object search/CMakeFiles/downward.dir/axioms.cc.o
[  1%] Building CXX object search/CMakeFiles/downward.dir/abstract_task.cc.o
[  1%] Built target translate
[  1%] Building CXX object search/CMakeFiles/downward.dir/command_line.cc.o
[  2%] Building CXX object search/CMakeFiles/downward.dir/evaluation_context.cc.o
[  2%] Building CXX object search/CMakeFiles/downward.dir/evaluation_result.cc.o
[  2%] Building CXX object search/CMakeFiles/downward.dir/evaluator.cc.o
[  3%] Building CXX object search/CMakeFiles

In [7]:
import sys

repo_path = str(REPO_DIR)
if repo_path not in sys.path:
    sys.path.insert(0, repo_path)

from src.data import find_pddl_files
from src.planner import run_fast_downward, read_plan

print("Project imports loaded.")


Project imports loaded.


In [8]:
KAGGLE_INPUT = Path("/kaggle/input")

if DATASET_ROOT is None:
    matches = list(
        KAGGLE_INPUT.rglob("blocks-strips-untyped/domain.pddl")
    )

    if not matches:
        raise FileNotFoundError(
            "Could not find blocks-strips-untyped/domain.pddl under /kaggle/input. "
            "Attach the IPC PDDL dataset using Add Input."
        )

    if len(matches) > 1:
        print("Multiple matching Blocksworld domains were found:")
        for match in matches:
            print(" -", match)
        print("\nUsing the first match. Set DATASET_ROOT manually if needed.")

    DOMAIN_FILE = matches[0]
    BLOCKS_DIR = DOMAIN_FILE.parent
else:
    BLOCKS_DIR = Path(DATASET_ROOT)
    DOMAIN_FILE = BLOCKS_DIR / "domain.pddl"

INSTANCE_DIR = BLOCKS_DIR / "instances"

if not DOMAIN_FILE.exists():
    raise FileNotFoundError(f"Domain file not found: {DOMAIN_FILE}")

if not INSTANCE_DIR.exists():
    raise FileNotFoundError(f"Instance directory not found: {INSTANCE_DIR}")

print("Domain:", DOMAIN_FILE)
print("Instances:", INSTANCE_DIR)


Domain: /kaggle/input/datasets/nikhilbathija/ipc-pddl-planning-instances/ipc-2000/domains/blocks-strips-untyped/domain.pddl
Instances: /kaggle/input/datasets/nikhilbathija/ipc-pddl-planning-instances/ipc-2000/domains/blocks-strips-untyped/instances


In [9]:
def instance_number(path):
    try:
        return int(path.stem.split("-")[-1])
    except ValueError:
        return path.stem

problems = sorted(
    INSTANCE_DIR.glob("*.pddl"),
    key=instance_number,
)

print("Number of problems:", len(problems))

for problem in problems[:10]:
    print(problem.name)


Number of problems: 102
instance-1.pddl
instance-2.pddl
instance-3.pddl
instance-4.pddl
instance-5.pddl
instance-6.pddl
instance-7.pddl
instance-8.pddl
instance-9.pddl
instance-10.pddl


In [10]:
if not problems:
    raise RuntimeError("No PDDL problem instances were found.")

first_problem = problems[0]
first_plan = WORKING_DIR / f"{first_problem.stem}_plan.txt"

result = run_fast_downward(
    fast_downward_path=fast_downward_script,
    domain_file=DOMAIN_FILE,
    problem_file=first_problem,
    plan_file=first_plan,
)

actions = read_plan(first_plan)

print("Problem:", first_problem.name)
print("Return code:", result["returncode"])
print("Plan found:", result["plan_found"])
print("Plan length:", len(actions))

for i, action in enumerate(actions, start=1):
    print(i, action)

if not result["plan_found"]:
    print("\nFast Downward output:")
    print(result["stdout"][-4000:])
    print(result["stderr"][-4000:])


Problem: instance-1.pddl
Return code: 0
Plan found: True
Plan length: 6
1 (pick-up b)
2 (stack b a)
3 (pick-up c)
4 (stack c b)
5 (pick-up d)
6 (stack d c)


In [11]:
import pandas as pd

rows = []

for problem in problems[:NUM_BASELINE_PROBLEMS]:
    plan_path = WORKING_DIR / f"{problem.stem}_plan.txt"

    # Remove an old plan with the same name so a failed run cannot
    # accidentally reuse a previous file.
    if plan_path.exists():
        plan_path.unlink()

    result = run_fast_downward(
        fast_downward_path=fast_downward_script,
        domain_file=DOMAIN_FILE,
        problem_file=problem,
        plan_file=plan_path,
    )

    actions = read_plan(plan_path)

    rows.append(
        {
            "problem": problem.name,
            "solved": bool(result["plan_found"]),
            "plan_length": len(actions),
            "returncode": result["returncode"],
        }
    )

    print(
        problem.name,
        "| solved:", result["plan_found"],
        "| plan length:", len(actions),
    )

baseline_df = pd.DataFrame(rows)
baseline_df


instance-1.pddl | solved: True | plan length: 6
instance-2.pddl | solved: True | plan length: 10
instance-3.pddl | solved: True | plan length: 6
instance-4.pddl | solved: True | plan length: 12
instance-5.pddl | solved: True | plan length: 10


,problem,solved,plan_length,returncode
0,instance-1.pddl,True,6,0
1,instance-2.pddl,True,10,0
2,instance-3.pddl,True,6,0
3,instance-4.pddl,True,12,0
4,instance-5.pddl,True,10,0


In [12]:
RESULT_PATH = WORKING_DIR / "blocksworld_fast_downward_baseline.csv"

baseline_df.to_csv(RESULT_PATH, index=False)

print("Saved:", RESULT_PATH)
print()
print(baseline_df.to_string(index=False))


Saved: /kaggle/working/blocksworld_fast_downward_baseline.csv

        problem  solved  plan_length  returncode
instance-1.pddl    True            6           0
instance-2.pddl    True           10           0
instance-3.pddl    True            6           0
instance-4.pddl    True           12           0
instance-5.pddl    True           10           0
